In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import os
import json

def load_datasets(path_697: str, path_320: str):
    df697 = pd.read_csv(path_697, sep=";")
    df320 = pd.read_csv(path_320, sep=";")
    print(f"[697] Loaded {df697.shape[0]} rows, {df697.shape[1]} columns")
    print(f"[320] Loaded {df320.shape[0]} rows, {df320.shape[1]} columns")
    return df697, df320


def engineer_697(df: pd.DataFrame):
    out = pd.DataFrame()

    grade_cols = [c for c in df.columns if "grade" in c.lower()]
    if grade_cols:
        raw_avg = df[grade_cols].mean(axis=1)
        out["previous_grade_score"] = (raw_avg / 20 * 100).clip(0, 100)
    else:
        out["previous_grade_score"] = np.nan

    admission_cols = [c for c in df.columns if "admission grade" in c.lower()]
    if admission_cols:
        col = admission_cols[0]
        out["background_academic_score"] = (df[col] / 200 * 100).clip(0, 100)
    else:
        out["background_academic_score"] = np.nan

    enroll_cols = [c for c in df.columns if "enrolled" in c.lower()]
    if enroll_cols:
        out["enrolled_units_count"] = df[enroll_cols].sum(axis=1).clip(3, 14)
    else:
        out["enrolled_units_count"] = np.nan

    fails_cols = [c for c in df.columns if "without evaluations" in c.lower()]
    if fails_cols:
        out["past_failures_count"] = df[fails_cols].sum(axis=1).clip(0, 10)
    else:
        out["past_failures_count"] = np.nan

    approved_cols = [c for c in df.columns if "approved" in c.lower()]
    if approved_cols:
        out["approved_units_count"] = df[approved_cols].sum(axis=1).clip(0, 30)
    else:
        out["approved_units_count"] = np.nan

    out["study_time_level_raw"] = np.nan
    out["absence_rate_raw"] = np.nan

    age_cols = [c for c in df.columns if "age at enrollment" in c.lower()]
    if age_cols:
        out["age"] = df[age_cols[0]].clip(17, 60)
    else:
        out["age"] = np.nan

    edu_cols = [
        c for c in df.columns
        if "education" in c.lower()
        and ("mother" in c.lower() or "father" in c.lower())
    ]

    if edu_cols:
        out["parent_education_raw"] = df[edu_cols].mean(axis=1)
    else:
        out["parent_education_raw"] = np.nan

    if "Target" in df.columns:
        out["student_status"] = df["Target"]
    else:
        out["student_status"] = np.nan

    out["source"] = "697"
    return out


def engineer_320(df: pd.DataFrame):
    out = pd.DataFrame()

    if "G1" in df.columns and "G2" in df.columns:
        out["previous_grade_score"] = ((df["G1"] + df["G2"]) / 2 / 20 * 100).clip(0, 100)
    else:
        out["previous_grade_score"] = np.nan

    out["background_academic_score"] = np.nan
    out["enrolled_units_count"] = 5.0

    if "failures" in df.columns:
        out["past_failures_count"] = df["failures"].clip(0, 10)
    else:
        out["past_failures_count"] = 0

    out["approved_units_count"] = np.nan

    if "studytime" in df.columns:
        study_map = {1: 2.5, 2: 3.5, 3: 7.0, 4: 12.0}
        out["study_time_level_raw"] = df["studytime"].map(study_map).fillna(3.5)
    else:
        out["study_time_level_raw"] = 3.5

    if "absences" in df.columns:
        def map_absence(x):
            if x == 0:
                return 0
            elif x <= 10:
                return 1
            else:
                return 2

        out["absence_rate_raw"] = df["absences"].apply(map_absence)
    else:
        out["absence_rate_raw"] = 1

    if "age" in df.columns:
        out["age"] = df["age"].clip(15, 30)
    else:
        out["age"] = np.nan

    edu_cols = [c for c in df.columns if c in ["Medu", "Fedu"]]

    if edu_cols:
        out["parent_education_raw"] = df[edu_cols].mean(axis=1)
    else:
        out["parent_education_raw"] = np.nan

    if "G3" in df.columns:
        def derive_status(g3):
            if g3 < 10:
                return "Dropout"
            elif g3 < 15:
                return "Enrolled"
            else:
                return "Graduate"

        out["student_status"] = df["G3"].apply(derive_status)
    else:
        out["student_status"] = np.nan

    out["source"] = "320"
    return out


def merge_and_preprocess(df697_eng, df320_eng):
    merged = pd.concat([df697_eng, df320_eng], ignore_index=True)
    print(f"[Merge] Combined shape: {merged.shape}")

    numeric_cols = [
        "previous_grade_score",
        "background_academic_score",
        "enrolled_units_count",
        "past_failures_count",
        "approved_units_count",
        "study_time_level_raw",
        "age",
        "parent_education_raw"
    ]

    for col in numeric_cols:
        median_val = merged[col].median()
        n_missing = merged[col].isna().sum()
        merged[col] = merged[col].fillna(median_val)

        if n_missing > 0:
            print(f"Imputed {n_missing} missing values in '{col}' with median = {median_val:.2f}")

    merged["parent_education_level_norm"] = (
        merged["parent_education_raw"] / 6 * 100
    ).clip(0, 100)

    def categorize_difficulty(n):
        if n <= 4:
            return "Low"
        elif n <= 6:
            return "Medium"
        else:
            return "High"

    merged["difficulty_level"] = merged["enrolled_units_count"].apply(categorize_difficulty)

    def categorize_study(h):
        if h < 5:
            return "Low"
        elif h <= 10:
            return "Medium"
        else:
            return "High"

    merged["study_time_level"] = merged["study_time_level_raw"].apply(categorize_study)

    absence_text_map = {
        0: "Rare",
        1: "Sometimes",
        2: "Often"
    }

    merged["absence_rate"] = merged["absence_rate_raw"].map(absence_text_map).fillna("Sometimes")

    def categorize_age(a):
        if a <= 20:
            return "18-20"
        elif a <= 23:
            return "21-23"
        else:
            return "24+"

    merged["age_group"] = merged["age"].apply(categorize_age)


    difficulty_map = {
        "Low": 0,
        "Medium": 1,
        "High": 2
    }

    study_time_map = {
        "Low": 0,
        "Medium": 1,
        "High": 2
    }

    absence_map = {
        "Rare": 0,
        "Sometimes": 1,
        "Often": 2
    }

    age_map = {
        "18-20": 0,
        "21-23": 1,
        "24+": 2
    }

    merged["difficulty_level_enc"] = merged["difficulty_level"].map(difficulty_map)
    merged["study_time_level_enc"] = merged["study_time_level"].map(study_time_map)
    merged["absence_rate_enc"] = merged["absence_rate"].map(absence_map)
    merged["age_group_enc"] = merged["age_group"].map(age_map)

    print("Manual encoding applied:")
    print("difficulty_level:", difficulty_map)
    print("study_time_level:", study_time_map)
    print("absence_rate:", absence_map)
    print("age_group:", age_map)

    # Target encoding is okay with LabelEncoder.
    merged["student_status"] = merged["student_status"].astype(str).str.strip()

    target_le = LabelEncoder()
    merged["student_status_enc"] = target_le.fit_transform(
        merged["student_status"].fillna("Enrolled")
    )

    print(f"Target classes: {dict(zip(target_le.classes_, target_le.transform(target_le.classes_)))}")

    return merged, target_le


def build_final_df(merged):
    feature_cols = [
        "previous_grade_score",
        "background_academic_score",
        "enrolled_units_count",
        "difficulty_level_enc",
        "past_failures_count",
        "approved_units_count",
        "study_time_level_enc",
        "absence_rate_enc",
        "age_group_enc",
        "parent_education_level_norm",
        "student_status_enc"
    ]

    final = merged[feature_cols].copy()

    final = final.rename(columns={
        "difficulty_level_enc": "difficulty_level",
        "study_time_level_enc": "study_time_level",
        "absence_rate_enc": "absence_rate",
        "age_group_enc": "age_group",
        "parent_education_level_norm": "parent_education_level",
        "student_status_enc": "student_status"
    })

    print(f"\n[Final] Training dataframe shape: {final.shape}")
    print(f"[Final] Missing values:\n{final.isnull().sum()}")
    print(f"\n[Final] Class distribution:\n{final['student_status'].value_counts()}")
    print(f"\n[Final] Feature dtypes:\n{final.dtypes}")
    print(f"\n[Final] Sample rows:\n{final.head()}")

    return final

In [2]:
PATH_697 = "dataset_697.csv"
PATH_320 = "student-mat.csv"

df697_raw, df320_raw = load_datasets(PATH_697, PATH_320)

print("\n── Engineering Dataset 697 features ──")
df697_eng = engineer_697(df697_raw)

print("\n── Engineering Dataset 320 features ──")
df320_eng = engineer_320(df320_raw)

print("\n── Merging and preprocessing ──")
merged, target_le = merge_and_preprocess(df697_eng, df320_eng)
print("\n── Building final training dataframe ──")
final_df = build_final_df(merged)

final_df.head()

[697] Loaded 4424 rows, 37 columns
[320] Loaded 395 rows, 33 columns

── Engineering Dataset 697 features ──

── Engineering Dataset 320 features ──

── Merging and preprocessing ──
[Merge] Combined shape: (4819, 11)
Imputed 395 missing values in 'background_academic_score' with median = 63.05
Imputed 395 missing values in 'approved_units_count' with median = 10.00
Imputed 4424 missing values in 'study_time_level_raw' with median = 3.50
Imputed 4424 missing values in 'parent_education_raw' with median = 2.50
Manual encoding applied:
difficulty_level: {'Low': 0, 'Medium': 1, 'High': 2}
study_time_level: {'Low': 0, 'Medium': 1, 'High': 2}
absence_rate: {'Rare': 0, 'Sometimes': 1, 'Often': 2}
age_group: {'18-20': 0, '21-23': 1, '24+': 2}
Target classes: {'Dropout': np.int64(0), 'Enrolled': np.int64(1), 'Graduate': np.int64(2)}

── Building final training dataframe ──

[Final] Training dataframe shape: (4819, 11)
[Final] Missing values:
previous_grade_score         0
background_academic_sc

,previous_grade_score,background_academic_score,enrolled_units_count,difficulty_level,past_failures_count,approved_units_count,study_time_level,absence_rate,age_group,parent_education_level,student_status
0,100.0,63.65,3.0,0,0,0.0,0,1,0,41.666667,0
1,100.0,71.25,12.0,2,0,12.0,0,1,0,41.666667,2
2,100.0,62.40,12.0,2,0,0.0,0,1,0,41.666667,0
3,100.0,59.80,12.0,2,0,11.0,0,1,0,41.666667,2
4,100.0,70.75,12.0,2,0,11.0,0,1,2,41.666667,2


In [3]:
import os

os.makedirs("output", exist_ok=True)
final_df.to_csv("output/training_data.csv", index=False)